In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")


In [ ]:
from gradio_client import Client, handle_file

client = Client("khang119966/DeepSeek-OCR-DEMO")
result = client.predict(
		image=handle_file('2.png'),
		model_size="Gundam (Recommended)",
		task_type="📝 Free OCR",
		ref_text="Hello!!",
		api_name="/process_ocr_task"
)
raw_text=result[0]
print(raw_text)

In [ ]:
import os
import json
import logging
from typing import Dict, Any

import google.genai as genai


# ---------------------------------------------------------------------
# Logging (Colab-safe)
# ---------------------------------------------------------------------
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------
# Configuration (LAZY loading – Colab safe)
# ---------------------------------------------------------------------
class GeminiConfig:
    @staticmethod
    def api_key() -> str:
        key = os.getenv("GEMINI_API_KEY")
        if not key:
            raise RuntimeError("GEMINI_API_KEY is not configured")
        return key

    @staticmethod
    def model() -> str:
        return os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

    @staticmethod
    def system_prompt() -> str:
        return os.getenv(
            "GEMINI_SYSTEM_PROMPT",
            (
                "You are an AI that extracts structured data from medical prescriptions.\n"
                "Extract ONLY the following fields:\n"
                "- patient_name\n"
                "- doctor_name\n"
                "- symptoms\n"
                "- prescription\n"
                "- dosage\n"
                "- doctor_notes\n\n"
                "Rules:\n"
                "1. Return ONLY a valid JSON object.\n"
                "2. No explanations, no markdown.\n"
                "3. Use null if a field is missing."
            )
        )


# ---------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------
def _initialize_gemini_client() -> genai.Client:
    return genai.Client(api_key=GeminiConfig.api_key())


def _build_prompt(ocr_text: str) -> str:
    return f"""
Prescription Text:
{ocr_text}

Expected JSON format:
{{
  "patient_name": null,
  "doctor_name": null,
  "symptoms": null,
  "prescription": null,
  "dosage": null,
  "doctor_notes": null
}}
""".strip()


def _parse_json_safely(text: str) -> Dict[str, Any]:
    """
    Handles cases where Gemini wraps JSON in ```json blocks.
    """
    cleaned = text.strip()

    if cleaned.startswith("```"):
        cleaned = cleaned.removeprefix("```json").removeprefix("```")
        cleaned = cleaned.removesuffix("```").strip()

    return json.loads(cleaned)


# ---------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------

async def extract_structured_data(ocr_text: str) -> Dict[str, Any]:
    try:
        if not ocr_text or not ocr_text.strip():
            raise ValueError("OCR text is empty")

        client = _initialize_gemini_client()

        full_prompt = (
            f"{GeminiConfig.system_prompt()}\n\n"
            f"{_build_prompt(ocr_text)}"
        )

        response = await client.aio.models.generate_content(
            model=GeminiConfig.model(),
            contents=[
                {
                    "role": "user",
                    "parts": [{"text": full_prompt}],
                }
            ],
        )

        structured_data = _parse_json_safely(response.text)

        return {
            "success": True,
            "data": structured_data,
            "error": None,
        }

    except Exception as exc:
        logger.exception("Structured extraction failed")
        return {
            "success": False,
            "data": None,
            "error": str(exc),
        }


In [ ]:
result = await extract_structured_data(raw_text)
result